# Derivation of Kupiec's Proportion of Failures (POF) Test

## Preliminary Calculations and Assumptions

**Assumptions**:
- We define a violation indicator $I_t = \mathbb{1}_{L_t > VaR_\alpha}$ for each day $t$ in the backtesting window, where $L_t$ is the realized loss and $VaR_\alpha$ is the model's VaR forecast for that day.
- We assume $I_1, I_2, \dots, I_n$ are iid Bernoulli random variables under the null hypothesis that the model is correctly calibrated, i.e. $I_t \sim \text{Bernoulli}(p)$ with true violation probability $p = 1-\alpha$.
- Let $n_1 = \sum_{t=1}^n I_t$ denote the number of violations, $n_0 = n - n_1$ the number of non-violations, and $n = n_0+n_1$ the total number of observations.

**Setup**:
The joint likelihood of $I_1,\dots,I_n$ given $p$ is
$$L(p) = p^{n_1}(1-p)^{n_0}\quad(1)$$

We define two parameter spaces:
- **Null hypothesis** $H_0: p \in \Theta_0 = \{1-\alpha\}$, a single fixed point (the model's claimed violation rate), so $r_0 = 0$ free parameters.
- **Unrestricted (alternative) hypothesis**: $p \in \Theta = (0,1)$, so $r=1$ free parameter.

## Deriving the MLE Under $\Theta$

We maximize the log-likelihood of (1) over $p\in(0,1)$. Taking the log,
$$\ell(p) = n_1\ln p + n_0\ln(1-p)$$

Differentiating and setting to zero,
$$\frac{d\ell}{dp} = \frac{n_1}{p} - \frac{n_0}{1-p} = 0$$

Solving,
$$n_1(1-p) = n_0 p \implies n_1 = p(n_0+n_1) = pn$$

$$\hat p = \frac{n_1}{n}\quad(2)$$

which is the observed violation rate — the natural estimator.

## Building the Likelihood Ratio

Since $\Theta_0$ is a single point, the maximum of $L(p)$ over $\Theta_0$ is just $L(1-\alpha)$ evaluated directly. The maximum over $\Theta$ is $L(\hat p)$ using (2). The likelihood ratio is
$$\lambda = \frac{\max_{p\in\Theta_0}L(p)}{\max_{p\in\Theta}L(p)} = \frac{L(1-\alpha)}{L(\hat p)} = \frac{(1-\alpha)^{n_1}\alpha^{n_0}}{\hat p^{n_1}(1-\hat p)^{n_0}}\quad(3)$$

## Applying the Asymptotic Theorem

By the likelihood ratio theorem (Wilks' theorem), since $\Theta$ has $r=1$ free parameter and $\Theta_0$ has $r_0=0$, the test statistic $-2\ln\lambda$ is asymptotically $\chi^2$ with $r-r_0 = 1$ degree of freedom under $H_0$.

Taking $-2\ln$ of (3):
$$LR_{POF} = -2\left[n_1\ln(1-\alpha) + n_0\ln(\alpha) - n_1\ln\hat p - n_0\ln(1-\hat p)\right]$$

$$\boxed{LR_{POF} = -2\ln\left[\frac{(1-\alpha)^{n_1}\alpha^{n_0}}{\hat p^{n_1}(1-\hat p)^{n_0}}\right]\sim\chi^2_{(1)}}\quad(4)$$

We reject the null hypothesis (i.e. conclude the VaR model is miscalibrated) if $LR_{POF}$ exceeds the $\chi^2_{(1)}$ critical value at our chosen test size (e.g. $3.84$ at 5% significance).

# Derivation of Christoffersen's Independence Test

## Preliminary Calculations and Assumptions

**Assumptions**:
- We reuse the violation indicator $I_t = \mathbb{1}_{L_t > VaR_\alpha}$ from the Kupiec test.
- We now model $\{I_t\}$ as a first-order Markov chain, so the probability of a violation on day $t$ is allowed to depend on whether day $t-1$ was a violation:
$$\pi_0 = \mathbb{P}(I_t=1 \mid I_{t-1}=0), \qquad \pi_1 = \mathbb{P}(I_t=1\mid I_{t-1}=1)$$
- Let $n_{ij}$ denote the number of days where $I_{t-1}=i$ and $I_t=j$, for $i,j\in\{0,1\}$. Note $n_{i0}+n_{i1}$ is the total number of days following a day in state $i$.
- **Intuition**: if the model is well-calibrated and violations are genuinely independent events, then knowing yesterday's outcome should tell us nothing about today's violation probability, i.e. $\pi_0=\pi_1$. If instead violations cluster — a string of losses breaching VaR back-to-back during a crisis — that means $\pi_1 \gg \pi_0$, revealing a blind spot: the model's assumptions (e.g. constant volatility) break down precisely when volatility regimes shift, and this test is designed to catch exactly that failure mode.

## Setting Up the Likelihoods

The likelihood of the observed transition counts, treating each row of the transition matrix as its own Bernoulli process, is
$$L(\pi_0,\pi_1) = (1-\pi_0)^{n_{00}}\pi_0^{n_{01}}(1-\pi_1)^{n_{10}}\pi_1^{n_{11}}\quad(1)$$

We define two parameter spaces:
- **Unrestricted (alternative) hypothesis**: $(\pi_0,\pi_1)\in\Theta = (0,1)^2$, so $r=2$ free parameters.
- **Null hypothesis** $H_0: \pi_0=\pi_1=\pi$ (independence), so $(\pi_0,\pi_1)\in\Theta_0$ collapses to a single free parameter $\pi\in(0,1)$, giving $r_0=1$.

## Deriving the MLEs Under $\Theta$

Notice (1) factors into two separate pieces, one depending only on $\pi_0$ and one only on $\pi_1$. We can therefore maximize each independently, exactly as we did for Kupiec's single Bernoulli likelihood.

Taking the log of the $\pi_0$-piece,
$$\ell(\pi_0) = n_{01}\ln\pi_0 + n_{00}\ln(1-\pi_0)$$

Differentiating and setting to zero,
$$\frac{d\ell}{d\pi_0} = \frac{n_{01}}{\pi_0} - \frac{n_{00}}{1-\pi_0} = 0 \implies \hat\pi_0 = \frac{n_{01}}{n_{00}+n_{01}}\quad(2)$$

By an identical argument on the $\pi_1$-piece,
$$\hat\pi_1 = \frac{n_{11}}{n_{10}+n_{11}}\quad(3)$$

Both are just the observed violation rate conditional on yesterday's state — the natural estimators.

## Deriving the MLE Under $\Theta_0$

Under $H_0$, (1) collapses to
$$L(\pi) = (1-\pi)^{n_{00}+n_{10}}\pi^{n_{01}+n_{11}}$$

which is structurally identical to the single Bernoulli likelihood from Kupiec's derivation. By the same maximization,
$$\hat\pi = \frac{n_{01}+n_{11}}{n_{00}+n_{01}+n_{10}+n_{11}} = \frac{n_{01}+n_{11}}{n}\quad(4)$$

## Building the Likelihood Ratio

$$\lambda = \frac{\max_{\pi\in\Theta_0}L(\pi)}{\max_{(\pi_0,\pi_1)\in\Theta}L(\pi_0,\pi_1)} = \frac{(1-\hat\pi)^{n_{00}+n_{10}}\hat\pi^{n_{01}+n_{11}}}{(1-\hat\pi_0)^{n_{00}}\hat\pi_0^{n_{01}}(1-\hat\pi_1)^{n_{10}}\hat\pi_1^{n_{11}}}\quad(5)$$

## Applying the Asymptotic Theorem

By the same likelihood ratio theorem used for Kupiec, since $\Theta$ has $r=2$ free parameters and $\Theta_0$ has $r_0=1$, the statistic $-2\ln\lambda$ is asymptotically $\chi^2$ with $r-r_0=1$ degree of freedom under $H_0$.

$$LR_{ind} = -2\left[(n_{00}+n_{10})\ln(1-\hat\pi) + (n_{01}+n_{11})\ln\hat\pi - n_{00}\ln(1-\hat\pi_0) - n_{01}\ln\hat\pi_0 - n_{10}ln(1-\hat\pi_1) - n_{11}ln\hat\pi_1\right]$$
$$\boxed{LR_{ind} = -2\ln\left[\frac{(1-\hat\pi)^{n_{00}+n_{10}}\hat\pi^{n_{01}+n_{11}}}{(1-\hat\pi_0)^{n_{00}}\hat\pi_0^{n_{01}}(1-\hat\pi_1)^{n_{10}}\hat\pi_1^{n_{11}}}\right]\sim\chi^2_{(1)}}\quad(6)$$

We reject the null hypothesis of independence (i.e. conclude violations cluster, signaling a blind spot where the model's assumptions break down under stress) if $LR_{ind}$ exceeds the $\chi^2_{(1)}$ critical value at our chosen significance level.

Cross-referenced with Christoffersen, P. (1998), "Evaluating Interval Forecasts," *International Economic Review*, 39(4), pp. 841-862.